In [ ]:
import json
import torch

from pprint import pprint # para prints mais bonitos
from limpeza import *
from sentence_transformers import SentenceTransformer

In [81]:
# Abrindo o json
with open("dados/noticias_brutas.json","r",encoding="utf-8") as file:
    dados = json.load(file)

In [82]:
pprint(dados, width=100)

[{'data': '2023-08-02',
  'fonte': 'Banco Central do Brasil',
  'id': 1,
  'texto': '<p>Publicado em: 02/08/2023 às 20h15</p>\n'
           '\n'
           '\n'
           '<p>O <strong>Comitê de Política Monetária</strong> (Copom) do <strong>Banco Central do '
           'Brasil</strong> decidiu, por unanimidade, manter a taxa básica de juros (Selic) em '
           '13,75% ao ano. Esta é a quarta reunião consecutiva em que o comitê opta pela '
           'manutenção.</p>\n'
           '\n'
           '<br/><br/>\n'
           '\n'
           '<p>Segundo o comunicado oficial, o Copom avaliou que o ambiente inflacionário ainda '
           'exige cautela, embora o IPCA acumulado nos últimos doze meses tenha   apresentado '
           'desaceleração consistente.   O colegiado ponderou, ainda, os efeitos defasados da '
           'política monetária já em vigor sobre a atividade econômica.</p>\n'
           '\n'
           '<p>Leia o comunicado completo em <a '
           "href='https://

# Etapa 1 — Limpeza e Tratamento de Texto

In [83]:
limpar_json(dados)
pprint(dados, width=100)

[{'data': '2023-08-02',
  'fonte': 'Banco Central do Brasil',
  'id': 1,
  'texto': 'O Comitê de Política Monetária (Copom) do Banco Central do Brasil decidiu, por '
           'unanimidade, manter a taxa básica de juros (Selic) em 13,75% ao ano. Esta é a quarta '
           'reunião consecutiva em que o comitê opta pela manutenção.\n'
           'Segundo o comunicado oficial, o Copom avaliou que o ambiente inflacionário ainda exige '
           'cautela, embora o IPCA acumulado nos últimos doze meses tenha apresentado '
           'desaceleração consistente. O colegiado ponderou, ainda, os efeitos defasados da '
           'política monetária já em vigor sobre a atividade econômica.',
  'titulo': 'Copom mantém Selic em 13,75% ao ano pela quarta reunião consecutiva'},
 {'data': '2023-08-09',
  'fonte': 'IBGE',
  'id': 2,
  'texto': 'O Índice Nacional de Preços ao Consumidor Amplo (IPCA) ficou em 0,12% em julho de '
           '2023, o menor resultado para o mês desde 2006, quando havia

In [84]:
# salvando os dados localmente
with open("dados/noticias_limpas.json", "w",encoding="utf-8") as file:
    json.dump(dados, file,ensure_ascii=False,indent=2)

In [85]:
pprint(dados, width=100)

[{'data': '2023-08-02',
  'fonte': 'Banco Central do Brasil',
  'id': 1,
  'texto': 'O Comitê de Política Monetária (Copom) do Banco Central do Brasil decidiu, por '
           'unanimidade, manter a taxa básica de juros (Selic) em 13,75% ao ano. Esta é a quarta '
           'reunião consecutiva em que o comitê opta pela manutenção.\n'
           'Segundo o comunicado oficial, o Copom avaliou que o ambiente inflacionário ainda exige '
           'cautela, embora o IPCA acumulado nos últimos doze meses tenha apresentado '
           'desaceleração consistente. O colegiado ponderou, ainda, os efeitos defasados da '
           'política monetária já em vigor sobre a atividade econômica.',
  'titulo': 'Copom mantém Selic em 13,75% ao ano pela quarta reunião consecutiva'},
 {'data': '2023-08-09',
  'fonte': 'IBGE',
  'id': 2,
  'texto': 'O Índice Nacional de Preços ao Consumidor Amplo (IPCA) ficou em 0,12% em julho de '
           '2023, o menor resultado para o mês desde 2006, quando havia

In [86]:
# separando os textos para o modelo de embeddings
textos = [item["texto"] for item in dados]
textos

['O Comitê de Política Monetária (Copom) do Banco Central do Brasil decidiu, por unanimidade, manter a taxa básica de juros (Selic) em 13,75% ao ano. Esta é a quarta reunião consecutiva em que o comitê opta pela manutenção.\nSegundo o comunicado oficial, o Copom avaliou que o ambiente inflacionário ainda exige cautela, embora o IPCA acumulado nos últimos doze meses tenha apresentado desaceleração consistente. O colegiado ponderou, ainda, os efeitos defasados da política monetária já em vigor sobre a atividade econômica.',
 'O Índice Nacional de Preços ao Consumidor Amplo (IPCA) ficou em 0,12% em julho de 2023, o menor resultado para o mês desde 2006, quando havia recuado 0,04%. No acumulado do ano, o índice chega a 3,19%, e em doze meses, a 3,99%.\nO grupo de Alimentação e Bebidas foi o principal responsável pela desaceleração, com variação negativa de 0,07%. Já Transportes apresentaram alta de 0,39%, puxados pelo aumento nos preços dos combustíveis.',
 'O Produto Interno Bruto (PIB) d

# Etapa 2 — Geração de Embeddings

 Decidi comparar dois modelos: [paraphrase-multilingual-mpnet-base-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2) e [paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)

In [87]:
# Modelo Mpnet-base-v2
modelo_mpnet = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
modelo_mpnet_embeddings = modelo_mpnet.encode_document(textos)
modelo_mpnet_embeddings

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5638.38it/s]


array([[-0.08124313, -0.10893045, -0.01763957, ...,  0.05523699,
        -0.09044036,  0.05548149],
       [-0.04981953,  0.04832487, -0.02160634, ..., -0.00900564,
        -0.0148079 ,  0.01029835],
       [-0.05922996,  0.05123207, -0.02255038, ...,  0.00113573,
         0.04517382,  0.00500588],
       ...,
       [-0.09902201,  0.05406219, -0.02043637, ...,  0.00338449,
         0.00886938,  0.01611684],
       [-0.03425283,  0.05802324, -0.02205552, ...,  0.00265894,
         0.03060731,  0.00924797],
       [-0.05642449,  0.07278302, -0.02193773, ...,  0.00409152,
         0.01679504,  0.00223056]], shape=(19, 768), dtype=float32)

In [88]:
# Modelo MiniLM-L12-v2
modelo_minilm = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
modelo_minilm_embeddings = modelo_minilm.encode_document(textos)
modelo_minilm_embeddings

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8651.99it/s]


array([[ 0.03046242,  0.13011755, -0.13940868, ..., -0.05350263,
         0.0148607 ,  0.00676435],
       [ 0.03688167,  0.00864389,  0.01653373, ..., -0.00741336,
        -0.06883686,  0.04763566],
       [-0.0276428 , -0.08240549, -0.02119488, ..., -0.03036806,
         0.02475522,  0.017866  ],
       ...,
       [-0.02559752, -0.02924811, -0.09865288, ..., -0.00330122,
        -0.11651989,  0.0524526 ],
       [-0.04770337, -0.11821681, -0.04540889, ...,  0.00038767,
         0.02828123, -0.00044989],
       [-0.03714328, -0.02416075, -0.05105674, ..., -0.0016683 ,
         0.00892248,  0.01997686]], shape=(19, 384), dtype=float32)

# Etapa 3 — Motor de Busca Semântico

In [89]:
buscas = [
    "mudanças na taxa de juros",
    "mercado de trabalho e desemprego",
    "inflação e preços ao consumidor",
]

Par o seguinte código usei como referência o código presente nesse site pois achei ele muito bom: https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html

In [90]:
# Modelo Mpnet-base-v2
top_k = min(5, len(dados))
for busca in buscas:
    busca_embedding = modelo_mpnet.encode_query(busca)

    escore_similaridade = modelo_mpnet.similarity(busca_embedding, modelo_mpnet_embeddings)[0]
    escores, indices = torch.topk(escore_similaridade, k=top_k)

    print("\nBusca:", busca)
    print("\nTop 5 artigos mais similares com a busca:")

    for escore, idx in zip(escores, indices):
        print(f"(Escore: {escore:.4f}) Id: {dados[idx]["id"]} | Prévia: {textos[idx][:80]}...")


Busca: mudanças na taxa de juros

Top 5 artigos mais similares com a busca:
(Escore: 0.6504) Id: 11 | Prévia: Economistas consultados pelo Banco Central no Relatório Focus revisaram para bai...
(Escore: 0.6224) Id: 1 | Prévia: O Comitê de Política Monetária (Copom) do Banco Central do Brasil decidiu, por u...
(Escore: 0.6114) Id: 10 | Prévia: O estoque total de crédito no sistema financeiro nacional atingiu R$ 5,6 trilhõe...
(Escore: 0.5999) Id: 6 | Prévia: Em movimento amplamente esperado pelo mercado financeiro, o Copom reduziu a taxa...
(Escore: 0.5800) Id: 16 | Prévia: O Índice Geral de Preços — Mercado (IGP-M) registrou variação de -0,53% em agost...

Busca: mercado de trabalho e desemprego

Top 5 artigos mais similares com a busca:
(Escore: 0.7789) Id: 14 | Prévia: Apesar da melhora generalizada no mercado de trabalho, a taxa de desemprego entr...
(Escore: 0.6897) Id: 4 | Prévia: A taxa de desemprego no Brasil recuou para 7,9% no segundo trimestre de 2023, o ...
(Escore: 0.4817)

In [91]:
# Modelo MiniLM-L12-v2
top_k = min(5, len(dados))
for busca in buscas:
    busca_embedding = modelo_minilm.encode_query(busca)

    escore_similaridade = modelo_minilm.similarity(busca_embedding, modelo_minilm_embeddings)[0]
    escores, indices = torch.topk(escore_similaridade, k=top_k)

    print("\nBusca:", busca)
    print("\nTop 5 artigos mais similares com a busca:")

    for escore, idx in zip(escores, indices):
        print(f"(Escore: {escore:.4f}) Id: {dados[idx]["id"]} | Prévia: {textos[idx][:80]}...")


Busca: mudanças na taxa de juros

Top 5 artigos mais similares com a busca:
(Escore: 0.6534) Id: 1 | Prévia: O Comitê de Política Monetária (Copom) do Banco Central do Brasil decidiu, por u...
(Escore: 0.6377) Id: 10 | Prévia: O estoque total de crédito no sistema financeiro nacional atingiu R$ 5,6 trilhõe...
(Escore: 0.6208) Id: 11 | Prévia: Economistas consultados pelo Banco Central no Relatório Focus revisaram para bai...
(Escore: 0.5785) Id: 6 | Prévia: Em movimento amplamente esperado pelo mercado financeiro, o Copom reduziu a taxa...
(Escore: 0.5714) Id: 15 | Prévia: O real acumulou valorização de 6,8% frente ao dólar no mês de agosto, tornando-s...

Busca: mercado de trabalho e desemprego

Top 5 artigos mais similares com a busca:
(Escore: 0.6656) Id: 14 | Prévia: Apesar da melhora generalizada no mercado de trabalho, a taxa de desemprego entr...
(Escore: 0.6214) Id: 4 | Prévia: A taxa de desemprego no Brasil recuou para 7,9% no segundo trimestre de 2023, o ...
(Escore: 0.4570)

No geral, o **paraphrase-multilingual-mpnet-base-v2** parece ter se destacado melhor no geral! Explicado melhor no ```README.md```